In [14]:
import os
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv
import mysql.connector
from sentence_transformers import SentenceTransformer
import numpy as np
import re
import faiss
import json
# === 1️⃣ Load cấu hình .env ===
load_dotenv(r"C:\Code\Ky5\SEG\Crawl\database\.env")

MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost").strip()
MYSQL_USER = os.getenv("MYSQL_USER", "root").strip()
MYSQL_PASSWORD = quote_plus(os.getenv("MYSQL_PASSWORD", "").strip())  # encode @
MYSQL_DB = os.getenv("MYSQL_DB", "VNLawsv3").strip()
MYSQL_TABLE = "law_chunks"

# === 2️⃣ Tạo SQLAlchemy engine ===
engine_url = f"mysql+mysqlconnector://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}/{MYSQL_DB}?charset=utf8mb4"
engine = create_engine(engine_url)

# === 3️⃣ Đọc dữ liệu trực tiếp từ MySQL ===
query = f"""
SELECT law_id, chunk_num, document_type, issuing_agency, issue_date,
       title, source_url, raw_title, category, chunk
FROM {MYSQL_TABLE}
"""

df = pd.read_sql(query, engine)

# === 4️⃣ Làm sạch & kiểm tra ===
df["chunk_num"] = pd.to_numeric(df["chunk_num"], errors="coerce").fillna(0).astype(int)

print("✅ Dữ liệu đã tải:", df.shape)
df.head(3)


✅ Dữ liệu đã tải: (374251, 10)


,law_id,chunk_num,document_type,issuing_agency,issue_date,title,source_url,raw_title,category,chunk
0,22714,1,NGHỊ ĐỊNH,BỘ TRƯỞNG BỘ QUỐC GIA GIÁO DỤC BAN HÀNH,1945-10-15,Nghị định năm 1945 về Hội đồng cố vấn học chín...,https://thuvienphapluat.vn/van-ban/Giao-duc/Ng...,Nghị định năm 1945 về Hội đồng cố vấn học chín...,Giáo dục,BỘ QUỐC GIA GIÁO DỤC VIỆT NAM DÂN CHỦ CỘNG HÒA...
1,22714,2,NGHỊ ĐỊNH,BỘ TRƯỞNG BỘ QUỐC GIA GIÁO DỤC BAN HÀNH,1945-10-15,Nghị định năm 1945 về Hội đồng cố vấn học chín...,https://thuvienphapluat.vn/van-ban/Giao-duc/Ng...,Nghị định năm 1945 về Hội đồng cố vấn học chín...,Giáo dục,TRƯỞNG BỘ QUỐC GIA GIÁO DỤC Chiếu chỉ Sắc lệnh...
2,22714,3,NGHỊ ĐỊNH,BỘ TRƯỞNG BỘ QUỐC GIA GIÁO DỤC BAN HÀNH,1945-10-15,Nghị định năm 1945 về Hội đồng cố vấn học chín...,https://thuvienphapluat.vn/van-ban/Giao-duc/Ng...,Nghị định năm 1945 về Hội đồng cố vấn học chín...,Giáo dục,đồng cố vấn học chính họp mỗi năm hai kỳ vào đ...


In [16]:
# Khởi tạo model E5-base (đa ngôn ngữ)
model_name = "intfloat/multilingual-e5-base"
model = SentenceTransformer(model_name, device="cuda")  # đổi "cuda" nếu có GPU

# Tiền xử lý nhẹ: chuẩn hóa khoảng trắng
def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.replace("\r", "\n")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Chuẩn bị passages theo format của E5
passages = ["passage: " + normalize_text(t) for t in df["chunk"].fillna("").tolist()]

# Encode theo batch (tùy RAM bạn có thể tăng/giảm batch_size)
embeddings = model.encode(
    passages,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True  # E5 khuyến nghị normalize khi dùng cosine/IP
)
embeddings = np.asarray(embeddings, dtype="float32")
embeddings.shape


Batches: 100%|██████████| 5848/5848 [1:02:01<00:00,  1.57it/s]


(374251, 768)

In [17]:


dim = embeddings.shape[1]  # 768 với e5-base
index = faiss.IndexFlatIP(dim)  # Inner Product (cosine nếu vector đã normalize)
index.add(embeddings)

# Lưu index + metadata mapping
os.makedirs("faiss_store", exist_ok=True)
faiss.write_index(index, "faiss_store/e5base_ip.index")

# Lưu map id → (law_id, row_index) để lần sau chỉ cần load
id_map = {
    int(i): {
        "row_id": int(i),
        "law_id": str(df.iloc[i]["law_id"]),
        "chunk_num": int(df.iloc[i]["chunk_num"])
    }
    for i in range(len(df))
}
with open("faiss_store/id_map.json", "w", encoding="utf-8") as f:
    json.dump(id_map, f, ensure_ascii=False, indent=2)

print("✅ FAISS index saved:", "faiss_store/e5base_ip.index")
print("✅ id_map saved:", "faiss_store/id_map.json")


✅ FAISS index saved: faiss_store/e5base_ip.index
✅ id_map saved: faiss_store/id_map.json


In [ ]:
# Load lại index (nếu cần ở phiên khác)
# index = faiss.read_index("faiss_store/e5base_ip.index")

In [20]:
from IPython.display import display, HTML

def search_chunks(query: str, top_k: int = 5):
    """Trả về top_k chunk khớp nhất với truy vấn."""
    q = "query: " + normalize_text(query)
    q_emb = model.encode([q], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")

    scores, idxs = index.search(q_emb, top_k)
    idxs = idxs[0]
    scores = scores[0]

    results = df.iloc[idxs].copy()
    results["score"] = scores
    cols = ["score","law_id","chunk_num","title","category","document_type","issuing_agency","source_url","chunk"]
    display(HTML(results[cols].to_html(escape=False)))
    return results


def search_grouped_full_doc(query: str, top_k_docs: int = 3, top_k_chunks: int = 50):
    """
    1) Lấy top_k_chunks chunks tốt nhất,
    2) Gom theo law_id, lấy max score làm score văn bản,
    3) Chọn top_k_docs văn bản,
    4) Trả lại FULL văn bản (gộp các chunk theo chunk_num) + metadata.
    """
    # Step 1: lấy nhiều chunk ứng viên
    q = "query: " + normalize_text(query)
    q_emb = model.encode([q], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")

    scores, idxs = index.search(q_emb, top_k_chunks)
    idxs = idxs[0]
    scores = scores[0]

    cand = df.iloc[idxs].copy()
    cand["score"] = scores

    # Step 2: gom theo law_id
    grouped = (
        cand.groupby("law_id", as_index=False)
            .agg(max_score=("score","max"),
                 title=("title","first"),
                 category=("category","first"),
                 document_type=("document_type","first"),
                 issuing_agency=("issuing_agency","first"),
                 source_url=("source_url","first"))
            .sort_values("max_score", ascending=False)
            .head(top_k_docs)
    )

    full_docs = []
    for _, row in grouped.iterrows():
        lid = row["law_id"]

        # Lấy toàn bộ chunk của văn bản này và lọc bỏ None / trống
        doc_chunks = (
            df[df["law_id"] == lid]
            .sort_values("chunk_num")["chunk"]
            .dropna()
            .tolist()
        )
        doc_chunks = [c for c in doc_chunks if isinstance(c, str) and c.strip()]
        full_text = "\n\n".join(doc_chunks) if doc_chunks else "(Không có nội dung hợp lệ)"

        full_docs.append({
            "law_id": lid,
            "score": float(row["max_score"]),
            "title": row["title"],
            "category": row["category"],
            "document_type": row["document_type"],
            "issuing_agency": row["issuing_agency"],
            "source_url": row["source_url"],
            "full_text": full_text
        })

    # Hiển thị tóm tắt kết quả
    show = pd.DataFrame([{
        "score": d["score"],
        "law_id": d["law_id"],
        "title": d["title"],
        "category": d["category"],
        "document_type": d["document_type"],
        "issuing_agency": d["issuing_agency"],
        "source_url": d["source_url"],
        "length_chars": len(d["full_text"])
    } for d in full_docs])

    print("🔎 Top văn bản (grouped by law_id):")
    display(HTML(show.to_html(escape=False)))

    return full_docs


In [21]:
# Trả về chunk tốt nhất
_ = search_chunks("nghị định về giáo dục", top_k=5)

# Trả về FULL văn bản (gom chunk theo law_id)
docs = search_grouped_full_doc("nghị định về giáo dục", top_k_docs=2, top_k_chunks=100)

# In thử full text của kết quả đầu tiên
print("\n===== FULL VĂN BẢN 1 =====")
print("Law ID:", docs[0]["law_id"])
print("Title:", docs[0]["title"])
print("URL:", docs[0]["source_url"])
print("\n--- Nội dung (rút gọn 1500 ký tự) ---\n")
print(docs[0]["full_text"][:1500], " ...")


,score,law_id,chunk_num,title,category,document_type,issuing_agency,source_url,chunk
182461,0.876164,44867,5,"Thông báo 199/1998/TB-VPCP về ý kiến của Phó Thủ tướng Phạm Gia Khiêm tại cuộc họp chuẩn bị Nghị định phân cấp quản lý giáo dục, đào tạo do Văn phòng Chính phủ ban hành",Bộ máy hành chính,NGHỊ ĐỊNH,CHÍNH PHỦ,https://thuvienphapluat.vn/van-ban/Bo-may-hanh-chinh/Thong-bao-199-1998-TB-VPCP-y-kien-Pho-Thu-tuong-Pham-Gia-Khiem-cuoc-hop-chuan-bi-Nghi-dinh-phan-cap-quan-ly-giao-duc-dao-tao-44867.aspx,"đào tạo, đồng thời nâng cao trách nhiệm của Ủy ban nhân dân các cấp, các ngành và đẩy mạnh công tác xã hội hóa trong giáo dục, đào tạo. Nội dung của Nghị định phải phù hợp với các văn bản pháp luật hiện hành, đồng thời nêu rõ trách nhiệm cụ thể trong quản lý giáo dục - đào tạo của các cơ quan nhà nước. - Cần làm rõ nội dung quản lý giáo dục - đào tạo, trên cơ sở đó xác định hợp lý bố cục và tên của Nghị định."
7295,0.872649,21760,2,Nghị định 1090-TTg năm 1956 về thành lập tại bộ Giáo dục một Vụ Tổ chức và Cán bộ do Thủ Tướng Chính Phủ ban hành.,Bộ máy hành chính,NGHỊ ĐỊNH,BỘ GIÁO DỤC MỘT VỤ TỔ CHỨC VÀ CÁN BỘ DO THỦ TƯỚNG CHÍNH PHỦ BAN HÀNH,https://thuvienphapluat.vn/van-ban/Bo-may-hanh-chinh/Nghi-dinh-1090-TTg-thanh-lap-tai-bo-Giao-duc-mot-Vu-To-chuc-va-Can-bo-21760.aspx,NGHỊ ĐỊNH: Điều 1. – Nay thành lập tại Bộ Giáo dục một Vụ Tổ chức và Cán bộ thay cho Phòng Tổ chức và Cán bộ cũ. Điều 2. – Vụ Tổ chức và Cán bộ của Bộ Giáo dục có nhiệm vụ lãnh đạo về mọi mặt công tác tổ chức và cán bộ trong ngành giáo dục. Điều 3. – Chi tiết thi hành nghị định này do ông Bộ trưởng Bộ Giáo dục quy định. Điều 4. – Ông Bộ trưởng Bộ Giáo dục chịu trách nhiệm thi hành nghị định này. K.T. THỦ TƯỚNG CHÍNH PHỦ PHÓ THỦ TƯỚNG Phan Kế Toại Văn bản này chưa cập nhật nội dung Tiếng Anh
182462,0.869825,44867,6,"Thông báo 199/1998/TB-VPCP về ý kiến của Phó Thủ tướng Phạm Gia Khiêm tại cuộc họp chuẩn bị Nghị định phân cấp quản lý giáo dục, đào tạo do Văn phòng Chính phủ ban hành",Bộ máy hành chính,NGHỊ ĐỊNH,CHÍNH PHỦ,https://thuvienphapluat.vn/van-ban/Bo-may-hanh-chinh/Thong-bao-199-1998-TB-VPCP-y-kien-Pho-Thu-tuong-Pham-Gia-Khiem-cuoc-hop-chuan-bi-Nghi-dinh-phan-cap-quan-ly-giao-duc-dao-tao-44867.aspx,"nhà nước. - Cần làm rõ nội dung quản lý giáo dục - đào tạo, trên cơ sở đó xác định hợp lý bố cục và tên của Nghị định. Phạm vi điều chỉnh của Nghị định bao gồm Giáo dục phổ thông, Giáo dục chuyên nghiệp, Đại học và Dạy nghề, chính quy và không chính quy."
8462,0.869219,21553,9,"Nghị định 199-NĐ năm 1957 quy định phụ cấp giảng dạy cho những giáo viên, cán bộ dạy các lớp bổ túc văn hóa tổ chức ngoài giờ làm việc cho cán bộ công nhân viên cơ quan, xí nghiệp quốc doanh, nông lâm trường quốc doanh và Công trường xây dựng cơ bản do Bộ trưởng Bộ Giáo Dục ban hành.",Lao động - Tiền lương,NGHỊ ĐỊNH,BỘ DẠY CÁC LỚP BỔ TÚC VĂN HÓA TỔ CHỨC NGOÀI GIỜ LÀM VIỆC CHO CÁN BỘ CÔNG NHÂN VIÊN CƠ QUAN,https://thuvienphapluat.vn/van-ban/Lao-dong-Tien-luong/Nghi-dinh-199-ND-quy-dinh-phu-cap-giao-vien-can-bo-day-bo-tuc-van-hoa-to-chuc-ngoai-gio-lam-viec-cho-can-bo-cong-nhan-vien-21553.aspx,"do ngân sách các ngành giáo dục ở trung ương và địa phương đài thọ. Điều 6. – Nghị định này áp dụng từ mồng 01 tháng 03 năm 1957. Một thông tư của Bộ Giáo dục sẽ ấn định rõ thể thức thi hành Nghị định này. Điều 7. – Các ông Chánh Văn phòng Bộ Giáo dục, Giám đốc Nha Bình dân học vụ, Chủ tịch Ủy ban Hành chính các liên khu, khu, thành phố Hà Nội, Hải Phòng và các tỉnh trực thuộc chiếu Nghị định thi hành. BỘ TRƯỞNG BỘ GIÁO DỤC Nguyễn Văn Huyên Văn bản này chưa cập nhật nội dung Tiếng Anh"
6329,0.866942,22905,14,nghị định 286-NĐ năm 1956 về việc ấn định số giờ dạy học tối đa hàng tuần của giáo viên các cấp do Bộ trưởng Bộ Giáo dục ban hành,Giáo dục,NGHỊ ĐỊNH,BỘ TRƯỞNG BỘ GIÁO DỤC BAN HÀNH,https://thuvienphapluat.vn/van-ban/Giao-duc/nghi-dinh-286-ND-an-dinh-so-gio-day-hoc-toi-da-hang-tuan-cua-giao-vien-cac-cap-22905.aspx,"thuộc. Điều 9. Nghị định này áp dụng chung cho tất cả giáo viên phổ thông, trung, ti

🔎 Top văn bản (grouped by law_id):


,score,law_id,title,category,document_type,issuing_agency,source_url,length_chars
0,0.876164,44867,"Thông báo 199/1998/TB-VPCP về ý kiến của Phó Thủ tướng Phạm Gia Khiêm tại cuộc họp chuẩn bị Nghị định phân cấp quản lý giáo dục, đào tạo do Văn phòng Chính phủ ban hành",Bộ máy hành chính,NGHỊ ĐỊNH,CHÍNH PHỦ,https://thuvienphapluat.vn/van-ban/Bo-may-hanh-chinh/Thong-bao-199-1998-TB-VPCP-y-kien-Pho-Thu-tuong-Pham-Gia-Khiem-cuoc-hop-chuan-bi-Nghi-dinh-phan-cap-quan-ly-giao-duc-dao-tao-44867.aspx,3015
1,0.872649,21760,Nghị định 1090-TTg năm 1956 về thành lập tại bộ Giáo dục một Vụ Tổ chức và Cán bộ do Thủ Tướng Chính Phủ ban hành.,Bộ máy hành chính,NGHỊ ĐỊNH,BỘ GIÁO DỤC MỘT VỤ TỔ CHỨC VÀ CÁN BỘ DO THỦ TƯỚNG CHÍNH PHỦ BAN HÀNH,https://thuvienphapluat.vn/van-ban/Bo-may-hanh-chinh/Nghi-dinh-1090-TTg-thanh-lap-tai-bo-Giao-duc-mot-Vu-To-chuc-va-Can-bo-21760.aspx,863



===== FULL VĂN BẢN 1 =====
Law ID: 44867
Title: Thông báo 199/1998/TB-VPCP về ý kiến của Phó Thủ tướng Phạm Gia Khiêm tại cuộc họp chuẩn bị Nghị định phân cấp quản lý giáo dục, đào tạo do Văn phòng Chính phủ ban hành
URL: https://thuvienphapluat.vn/van-ban/Bo-may-hanh-chinh/Thong-bao-199-1998-TB-VPCP-y-kien-Pho-Thu-tuong-Pham-Gia-Khiem-cuoc-hop-chuan-bi-Nghi-dinh-phan-cap-quan-ly-giao-duc-dao-tao-44867.aspx

--- Nội dung (rút gọn 1500 ký tự) ---

VĂN PHÒNG CHÍNH PHỦ CỘNG HOÀ XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc Số: 199/1998/TB-VPCP Hà Nội, ngày 01 tháng 12 năm 1998 THÔNG BÁO CỦA CHÍNH PHỦ SỐ 199/1998/TB-VPCP NGÀY 01 THÁNG 12 NĂM 1998 Ý KIẾN CỦA PHÓ THỦ TƯỚNG PHẠM GIA KHIÊM TẠI CUỘC HỌP CHUẨN BỊ NGHỊ ĐỊNH PHÂN CẤP QUẢN LÝ GIÁO DỤC ĐÀO TẠO Ngày 18 tháng 11 năm 1998, tại Văn phòng Chính phủ, Phó Thủ tướng Phạm Gia Khiêm đã chủ trì cuộc họp bàn về Nghị định phân cấp quản lý giáo dục đào tạo.

năm 1998, tại Văn phòng Chính phủ, Phó Thủ tướng Phạm Gia Khiêm đã chủ trì cuộc 

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss, json, pandas as pd

# Load model & index
model = SentenceTransformer("faiss_store/e5_base_model")
index = faiss.read_index("faiss_store/e5base_ip.index")

# Load metadata
with open("faiss_store/id_map.json", encoding="utf-8") as f:
    id_map = json.load(f)

print("✅ Model, Index và Metadata đã load xong!")


In [ ]:
search_chunks("nghị định giáo dục")
search_grouped_full_doc("chính sách y tế", top_k_docs=3)
